# 🚗 CarX: Used Car Price Prediction using XGBoost Regression
### End-to-End Supervised Continuous Regression, EDA, Feature Engineering, Explainability & Inference Pipeline

**Project:** CarX Valuation Intelligence Platform  
**Dataset:** Indian Secondary Automobile Market Listings (`data/cardekho_imputated.csv`)  
**Environment:** Python 3.10+ / Jupyter Notebook / Google Colab

---

## 1. Problem Definition

The **Used Car Price Prediction** task is a **supervised continuous regression problem** aimed at predicting the fair market value (`selling_price` in INR ₹) of a used automobile in India based on its physical attributes, usage history, and mechanical specifications.

Because the used car market is fragmented and pricing fluctuates based on **brand equity, vehicle age, engine capacity, fuel efficiency, and transmission type**, individual sellers and dealerships lack a standardized, data-driven pricing benchmark. 

**Core Objectives:**
1. Prevent prolonged listing times resulting from overpricing.
2. Eliminate financial loss and inventory depreciation resulting from underpricing.
3. Construct a high-accuracy, robust regression model using **Extreme Gradient Boosting (XGBoost)**.
4. Deliver an end-to-end, reproducible inference pipeline suitable for real-time web deployment.

---

## 2. Concepts Used

- **Supervised Continuous Regression**: Modeling non-linear relationships between multi-modal vehicle attributes and a continuous monetary target (`selling_price`).
- **Gradient Boosted Decision Trees (XGBoost)**: Fitting an ensemble of decision trees sequentially, optimizing an objective function with second-order Taylor expansion (gradients $g_i$ and hessians $h_i$) with $L_1$ and $L_2$ regularization to prevent overfitting.
- **Exploratory Data Analysis (EDA)**: Profiling distributions, examining target skewness, computing multi-collinearity matrices, analyzing vehicle age depreciation decay, and quantifying brand equity premiums.
- **Data Preprocessing & Feature Engineering**: Parsing string-formatted numeric representations (stripping units such as `cc`, `bhp`, `kmpl`), imputing missing mechanical attributes using brand/model segment medians, and applying categorical encoding.
- **Regression Metrics Evaluation**:
  - **Root Mean Squared Error (RMSE)**: Penalizes large outlier errors: $\text{RMSE} = \sqrt{\frac{1}{n} \sum_{i=1}^n (y_i - \hat{y}_i)^2}$
  - **Mean Absolute Error (MAE)**: Measures average absolute currency error in INR: $\text{MAE} = \frac{1}{n} \sum_{i=1}^n |y_i - \hat{y}_i|$
  - **Coefficient of Determination ($R^2$)**: Quantifies percentage of variance explained by features: $R^2 = 1 - \frac{\sum (y_i - \hat{y}_i)^2}{\sum (y_i - \bar{y})^2}$
- **Model Explainability & Inference Pipeline**: Extracting tree-based feature importances (Gini/Gain) and building a modular, reusable inference function `predict_car_price` for real-time sample predictions.

---

## 3. Tech Stack Used

- **Programming Language**: Python (3.10+)
- **Data Manipulation & Analysis**: `pandas`, `numpy`
- **Data Visualization**: `matplotlib`, `seaborn`
- **Machine Learning Frameworks**: `xgboost` (`XGBRegressor`), `scikit-learn` (`train_test_split`, `StandardScaler`, `OneHotEncoder`, `RandomForestRegressor`, `GradientBoostingRegressor`, `MAE`, `MSE`, `RMSE`, `R2 Score`)
- **Model Serialization**: `joblib`, `json` (pure-Python lightweight decision tree dumper for serverless environments)
- **Development Environment**: Jupyter Notebook / Google Colab

---

## 5. Data Description (Feature Information)

| Feature | Type | Description |
| :--- | :--- | :--- |
| **`car_name`** | String | Full name of the car, combining brand and specific model variant (e.g., *Maruti Alto 800*, *Hyundai Grand i10*). |
| **`brand`** | String / Categorical | Brand / Manufacturer of the automobile (e.g., *Maruti*, *Hyundai*, *Honda*, *Toyota*, *BMW*). |
| **`model`** | String / Categorical | Specific vehicle model name under the brand (e.g., *Swift*, *City*, *Creta*, *Fortuner*). |
| **`seller_type`** | Categorical | Type of seller listing the car (*Individual*, *Dealer*, *Trustmark Dealer*). |
| **`fuel_type`** | Categorical | Primary fuel used by the vehicle (*Petrol*, *Diesel*, *CNG*, *LPG*, *Electric*). |
| **`transmission_type`** | Categorical | Transmission mechanism (*Manual*, *Automatic*). |
| **`vehicle_age`** | Integer | Age of the automobile in years since original registration date. |
| **`km_driven`** | Integer / Float | Total cumulative distance driven by the car in kilometers. |
| **`mileage`** | Float | Certified fuel efficiency in kilometers per litre (kmpl) or km/kg for gas. |
| **`engine`** | Float | Engine displacement volume in cubic centimeters (cc). |
| **`max_power`** | Float | Maximum brake horsepower produced by the engine (BHP). |
| **`seats`** | Integer | Passenger seating capacity of the vehicle. |
| **`selling_price`** | Float (Target) | The historical transaction or listing price in Indian Rupees (INR ₹). |

---
## 4. Implementation Workflow

### Step 1: Import Necessary Libraries
Load core numerical computing, data analytics, visualization, and machine learning modules.

In [ ]:
import os
import sys
import re
import json
import math
from pathlib import Path
from datetime import datetime

# Numerical & Data Analysis
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning: Scikit-Learn
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

# Machine Learning: XGBoost
try:
    from xgboost import XGBRegressor
    HAS_XGBOOST = True
except ImportError:
    HAS_XGBOOST = False

# Setup visualization styles
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

print(f"Python Version: {sys.version.split()[0]}")
print(f"Pandas Version: {pd.__version__}")
print(f"NumPy Version: {np.__version__}")
print(f"XGBoost Available: {HAS_XGBOOST}")

---
### Step 2: Import the Dataset
Load the Indian used car dataset (`data/cardekho_imputated.csv`) containing vehicle attributes, physical usage, mechanical specifications, and selling prices.

In [ ]:
# Locate dataset path across working directory structures
candidate_paths = [
    Path('data/cardekho_imputated.csv'),
    Path('../data/cardekho_imputated.csv'),
    Path('/content/data/cardekho_imputated.csv'),
    Path('/content/cardekho_imputated.csv'),
]

dataset_path = None
for p in candidate_paths:
    if p.exists():
        dataset_path = p
        break

if dataset_path is None:
    raise FileNotFoundError("Please place 'cardekho_imputated.csv' in 'data/' or current working directory.")

df_raw = pd.read_csv(dataset_path)
if 'Unnamed: 0' in df_raw.columns:
    df_raw = df_raw.drop(columns=['Unnamed: 0'])

print(f"Dataset successfully loaded from: {dataset_path}")
print(f"Total Records: {df_raw.shape[0]:,}")
print(f"Total Attributes: {df_raw.shape[1]}")
print("\nFirst 5 Records:")
display(df_raw.head())

---
### Step 3: Exploratory Data Analysis (EDA)

#### 3.1 Data Types, Summary Statistics & Missing Values
Inspect data types, statistical distributions across continuous features, and verify completeness.

In [ ]:
print("--- Data Information & Types ---")
df_raw.info()

print("\n--- Missing Value Count per Feature ---")
print(df_raw.isnull().sum())

print("\n--- Descriptive Summary Statistics ---")
display(df_raw.describe().T)

#### 3.2 Target Variable Distribution (`selling_price`)
Used car prices exhibit strong right-skewness: most mass-market vehicles range between ₹1 Lakh and ₹15 Lakhs, while a long tail of luxury vehicles (Mercedes-Benz, BMW, Porsche) extends up to ₹3.9+ Crores.

In gradient boosted regression, training directly on heavily skewed monetary targets produces disproportionately large residuals on high-value outliers. Modeling on the log-transformed scale $\log(1 + \text{selling\_price})$ stabilizes the variance and optimizes relative percentage error.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Raw Selling Price in Lakhs
sns.histplot(df_raw['selling_price'] / 100000, kde=True, ax=axes[0], color='#2563eb', bins=50)
axes[0].set_title('Raw Selling Price Distribution (₹ Lakhs)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Price (₹ Lakhs)')
axes[0].set_ylabel('Number of Listings')
axes[0].set_xlim(0, 50)

# Log-Transformed Selling Price
sns.histplot(np.log1p(df_raw['selling_price']), kde=True, ax=axes[1], color='#16a34a', bins=50)
axes[1].set_title('Distribution of log1p(Selling Price)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('log(1 + Selling Price)')
axes[1].set_ylabel('Number of Listings')

plt.tight_layout()
plt.show()

#### 3.3 Vehicle Age Depreciation Decay & Usage Impact (`km_driven`)
Analyze how market value decays with chronological age and usage intensity across fuel types.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Vehicle Age vs Selling Price
sns.scatterplot(
    x='vehicle_age',
    y=df_raw['selling_price'] / 100000,
    hue='transmission_type',
    data=df_raw,
    ax=axes[0],
    alpha=0.6
)
axes[0].set_title('Selling Price vs. Vehicle Age (Years)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Vehicle Age (Years)')
axes[0].set_ylabel('Selling Price (₹ Lakhs)')
axes[0].set_ylim(0, 100)

# Odometer (km_driven) vs Selling Price
sns.scatterplot(
    x=df_raw['km_driven'] / 1000,
    y=df_raw['selling_price'] / 100000,
    hue='fuel_type',
    data=df_raw,
    ax=axes[1],
    alpha=0.6
)
axes[1].set_title('Selling Price vs. Distance Driven ('000 km)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Distance Driven ('000 km)')
axes[1].set_ylabel('Selling Price (₹ Lakhs)')
axes[1].set_xlim(0, 300)
axes[1].set_ylim(0, 100)

plt.tight_layout()
plt.show()

#### 3.4 Mechanical Attributes: Engine Displacement, Max Power & Mileage
Examine the physical determinants of vehicle value: high engine displacement (`engine` in cc) and high brake horsepower (`max_power` in BHP) correlate strongly with vehicle tier and asking price.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Max Power vs Selling Price
sns.scatterplot(x='max_power', y=df_raw['selling_price'] / 100000, data=df_raw, ax=axes[0], color='#8b5cf6', alpha=0.5)
axes[0].set_title('Price vs. Max Power (BHP)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Max Power (BHP)')
axes[0].set_ylabel('Selling Price (₹ Lakhs)')
axes[0].set_ylim(0, 150)

# Engine Capacity vs Selling Price
sns.scatterplot(x='engine', y=df_raw['selling_price'] / 100000, data=df_raw, ax=axes[1], color='#f97316', alpha=0.5)
axes[1].set_title('Price vs. Engine Displacement (cc)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Engine (cc)')
axes[1].set_ylabel('Selling Price (₹ Lakhs)')
axes[1].set_ylim(0, 150)

# Mileage vs Selling Price
sns.scatterplot(x='mileage', y=df_raw['selling_price'] / 100000, data=df_raw, ax=axes[2], color='#06b6d4', alpha=0.5)
axes[2].set_title('Price vs. Fuel Mileage (kmpl)', fontsize=12, fontweight='bold')
axes[2].set_xlabel('Mileage (kmpl)')
axes[2].set_ylabel('Selling Price (₹ Lakhs)')
axes[2].set_ylim(0, 150)

plt.tight_layout()
plt.show()

#### 3.5 Brand Equity Analysis (Top 15 Automotive Brands by Listing Volume)
Compare median selling prices across mainstream and premium brand segments in the Indian market.

In [ ]:
top_brands = df_raw['brand'].value_counts().head(15).index
brand_order = df_raw[df_raw['brand'].isin(top_brands)].groupby('brand')['selling_price'].median().sort_values(ascending=False).index

plt.figure(figsize=(16, 6))
sns.barplot(
    x='brand',
    y=df_raw['selling_price'] / 100000,
    data=df_raw[df_raw['brand'].isin(top_brands)],
    order=brand_order,
    estimator=np.median,
    palette='viridis',
    errorbar=None
)
plt.title('Median Selling Price by Top 15 Brands in India (₹ Lakhs)', fontsize=14, fontweight='bold')
plt.xlabel('Brand')
plt.ylabel('Median Price (₹ Lakhs)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

#### 3.6 Correlation Heatmap of Continuous Numerical Features

In [ ]:
num_cols = ['vehicle_age', 'km_driven', 'mileage', 'engine', 'max_power', 'seats', 'selling_price']
corr = df_raw[num_cols].corr()

plt.figure(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='Blues', vmin=-1, vmax=1, cbar_kws={'label': 'Pearson Correlation'})
plt.title('Feature Correlation Matrix', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
### Step 4: Data Preprocessing and Feature Engineering

1. **String-Numeric Cleaner**: Strip residual unit strings (`cc`, `bhp`, `kmpl`, `km/kg`).
2. **Missing Value Imputation**: Impute missing mechanical specifications using brand and model hierarchical medians.
3. **Categorical Feature Preparation**: Identify nominal categories (`brand`, `model`, `seller_type`, `fuel_type`, `transmission_type`).
4. **Train-Test Splitting**: Stratified 80/20 split to guarantee identical brand distribution across training and unseen test sets.
5. **Feature Scaling & One-Hot Encoding**: Pipeline with `StandardScaler` on numerical features and `OneHotEncoder(handle_unknown='ignore')` on categorical features.

In [ ]:
def clean_unit_values(val):
    """Extract numerical values from text containing units like cc, bhp, or kmpl."""
    if pd.isna(val) or val is None:
        return np.nan
    if isinstance(val, (int, float)):
        return float(val)
    s = str(val).strip().lower()
    s = re.sub(r'[^\d.]', '', s)
    try:
        return float(s)
    except ValueError:
        return np.nan

# Create clean working copy
df_clean = df_raw.copy()

# Apply string unit cleaner across numerical fields
df_clean['vehicle_age'] = df_clean['vehicle_age'].apply(clean_unit_values).fillna(5).astype(int)
df_clean['km_driven'] = df_clean['km_driven'].apply(clean_unit_values).fillna(50000).astype(int)
df_clean['mileage'] = df_clean['mileage'].apply(clean_unit_values)
df_clean['engine'] = df_clean['engine'].apply(clean_unit_values)
df_clean['max_power'] = df_clean['max_power'].apply(clean_unit_values)
df_clean['seats'] = df_clean['seats'].apply(clean_unit_values).fillna(5).astype(int)
df_clean['selling_price'] = df_clean['selling_price'].apply(clean_unit_values).astype(float)

# Impute missing mechanical features by brand & model median
for col in ['mileage', 'engine', 'max_power']:
    df_clean[col] = df_clean.groupby(['brand', 'model'])[col].transform(lambda x: x.fillna(x.median()))
    df_clean[col] = df_clean[col].fillna(df_clean[col].median())

# Replace seats <= 0 with default 5
df_clean.loc[df_clean['seats'] <= 0, 'seats'] = 5

# Filter out zero or negative selling prices
df_clean = df_clean[df_clean['selling_price'] > 10000].reset_index(drop=True)

NUMERICAL_FEATURES = ['vehicle_age', 'km_driven', 'mileage', 'engine', 'max_power', 'seats']
CATEGORICAL_FEATURES = ['brand', 'model', 'seller_type', 'fuel_type', 'transmission_type']

print(f"Preprocessed Dataset Cleaned. Total Verified Records: {len(df_clean):,}")
print(f"Numerical Features: {NUMERICAL_FEATURES}")
print(f"Categorical Features: {CATEGORICAL_FEATURES}")

#### 4.2 Train-Test Split (80% Train, 20% Unseen Test)

In [ ]:
train_df, test_df = train_test_split(
    df_clean,
    test_size=0.20,
    random_state=42,
    stratify=df_clean['brand'] if df_clean['brand'].value_counts().min() > 1 else None
)

print(f"Training Set Size:   {len(train_df):,} samples ({len(train_df)/len(df_clean)*100:.1f}%)")
print(f"Testing Set Size:    {len(test_df):,} samples ({len(test_df)/len(df_clean)*100:.1f}%)")

X_train = train_df[NUMERICAL_FEATURES + CATEGORICAL_FEATURES]
y_train = np.log1p(train_df['selling_price'])  # Log target for model fitting

X_test = test_df[NUMERICAL_FEATURES + CATEGORICAL_FEATURES]
y_test_actual = test_df['selling_price']      # Real INR prices for evaluation

# ColumnTransformer Preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), NUMERICAL_FEATURES),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), CATEGORICAL_FEATURES),
    ]
)

print("Preprocessor ColumnTransformer configured successfully.")

---
### Step 5: Model Development & Multi-Algorithm Benchmarking

We compare multiple regression paradigms on the exact same unseen test set:
1. **Baseline Median Model**: Hierarchical lookup `(brand, model, age) -> (brand, model) -> (brand) -> global median`
2. **Random Forest Regressor**: Non-linear ensemble of bagged decision trees
3. **Gradient Boosting Regressor**: Sequentially boosted decision trees with shrinkage
4. **XGBoost Regressor**: Optimized gradient boosting with second-order Taylor expansion and tree regularization

In [ ]:
# 1. Hierarchical Baseline Lookup
def build_baseline_lookup(df):
    lookup = {
        'by_brand_model_age': {},
        'by_brand_model': {},
        'by_brand': {},
        'global_median': float(df['selling_price'].median())
    }
    for (brand, model, age), g in df.groupby(['brand', 'model', 'vehicle_age']):
        lookup['by_brand_model_age'][f"{brand}|{model}|{age}"] = float(g['selling_price'].median())
    for (brand, model), g in df.groupby(['brand', 'model']):
        lookup['by_brand_model'][f"{brand}|{model}"] = float(g['selling_price'].median())
    for brand, g in df.groupby('brand'):
        lookup['by_brand'][str(brand)] = float(g['selling_price'].median())
    return lookup

baseline_table = build_baseline_lookup(train_df)

def predict_baseline(row):
    b, m, a = row['brand'], row['model'], row['vehicle_age']
    k3 = f"{b}|{m}|{a}"
    if k3 in baseline_table['by_brand_model_age']:
        return baseline_table['by_brand_model_age'][k3]
    k2 = f"{b}|{m}"
    if k2 in baseline_table['by_brand_model']:
        return max(50000.0, baseline_table['by_brand_model'][k2] * max(0.25, 1.0 - 0.07 * (a - 4)))
    if b in baseline_table['by_brand']:
        return max(50000.0, baseline_table['by_brand'][b] * max(0.25, 1.0 - 0.07 * (a - 4)))
    return baseline_table['global_median']

base_preds = test_df.apply(predict_baseline, axis=1)
base_mae = mean_absolute_error(y_test_actual, base_preds)
base_rmse = np.sqrt(mean_squared_error(y_test_actual, base_preds))
base_r2 = r2_score(y_test_actual, base_preds)

print(f"Baseline Median Model -> MAE: ₹{base_mae:,.2f} | RMSE: ₹{base_rmse:,.2f} | R2: {base_r2:.4f}")

#### 5.2 Training Random Forest & Tuned XGBoost Models

In [ ]:
candidate_models = {
    'Random Forest': RandomForestRegressor(
        n_estimators=100,
        max_depth=12,
        min_samples_split=4,
        random_state=42,
        n_jobs=-1
    ),
    'Gradient Boosting': GradientBoostingRegressor(
        n_estimators=120,
        learning_rate=0.08,
        max_depth=5,
        random_state=42
    )
}

if HAS_XGBOOST:
    candidate_models['XGBoost'] = XGBRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_alpha=0.05,
        reg_lambda=1.0,
        random_state=42,
        n_jobs=-1
    )

benchmark_results = {
    'Baseline Median': {'MAE (₹)': base_mae, 'RMSE (₹)': base_rmse, 'R² Score': base_r2}
}
trained_pipelines = {}
predictions_actual = {}

for name, regressor in candidate_models.items():
    print(f"Fitting {name} pipeline...")
    pipe = Pipeline(steps=[('preprocessor', preprocessor), ('regressor', regressor)])
    pipe.fit(X_train, y_train)
    
    # Predict log-price and invert with expm1
    preds_log = pipe.predict(X_test)
    preds_actual = np.expm1(preds_log)
    
    mae = mean_absolute_error(y_test_actual, preds_actual)
    rmse = np.sqrt(mean_squared_error(y_test_actual, preds_actual))
    r2 = r2_score(y_test_actual, preds_actual)
    
    benchmark_results[name] = {'MAE (₹)': mae, 'RMSE (₹)': rmse, 'R² Score': r2}
    trained_pipelines[name] = pipe
    predictions_actual[name] = preds_actual
    print(f"  -> {name} MAE: ₹{mae:,.2f} | RMSE: ₹{rmse:,.2f} | R²: {r2:.4f}")

# Display Model Comparison Table
df_comparison = pd.DataFrame(benchmark_results).T
display(df_comparison.style.format({'MAE (₹)': '₹{:,.2f}', 'RMSE (₹)': '₹{:,.2f}', 'R² Score': '{:.4f}'}))

#### 5.3 Parity Plot: Actual Selling Price vs. Predicted Selling Price (XGBoost)
The parity plot visualizes how tightly predictions cluster along the ideal $y = x$ diagonal line across all price tiers.

In [ ]:
best_algo = 'XGBoost' if 'XGBoost' in trained_pipelines else 'Random Forest'
best_preds = predictions_actual[best_algo]

plt.figure(figsize=(9, 8))
plt.scatter(y_test_actual / 100000, best_preds / 100000, alpha=0.35, color='#2563eb', edgecolors='none', s=25)

# Perfect prediction line
max_val = min(100, max(y_test_actual.max() / 100000, best_preds.max() / 100000))
plt.plot([0, max_val], [0, max_val], color='#ef4444', linestyle='--', linewidth=2, label='Ideal Fit (Actual = Predicted)')

plt.title(f'Actual vs. Predicted Selling Price ({best_algo})', fontsize=14, fontweight='bold')
plt.xlabel('Actual Price (₹ Lakhs)', fontsize=12)
plt.ylabel('Predicted Price (₹ Lakhs)', fontsize=12)
plt.xlim(0, max_val)
plt.ylim(0, max_val)
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()

---
### Step 6: Model Explainability (Tree Feature Importances)
Extract feature importances from the trained tree models (Random Forest and XGBoost) to reveal the dominant pricing drivers in the Indian automotive market.

In [ ]:
# Extract feature importances
pipe = trained_pipelines[best_algo]
reg = pipe.named_steps['regressor']
ohe = pipe.named_steps['preprocessor'].named_transformers_['cat']
cat_feature_names = ohe.get_feature_names_out(CATEGORICAL_FEATURES)
all_feature_names = NUMERICAL_FEATURES + list(cat_feature_names)
raw_importances = reg.feature_importances_

# Aggregate one-hot encoded categories back to feature domain groups
feature_groups = {f: 0.0 for f in NUMERICAL_FEATURES + CATEGORICAL_FEATURES}
for feat_name, imp in zip(all_feature_names, raw_importances):
    matched = False
    for cat_f in CATEGORICAL_FEATURES:
        if feat_name.startswith(f"{cat_f}_"):
            feature_groups[cat_f] += float(imp)
            matched = True
            break
    if not matched and feat_name in feature_groups:
        feature_groups[feat_name] += float(imp)

# Normalize to percentage
total_imp = sum(feature_groups.values())
importance_series = pd.Series({k: (v / total_imp) * 100 for k, v in feature_groups.items()}).sort_values(ascending=False)

plt.figure(figsize=(12, 6))
sns.barplot(x=importance_series.values, y=importance_series.index, palette='crest')
plt.title(f'Feature Importance Breakdown ({best_algo} Model)', fontsize=14, fontweight='bold')
plt.xlabel('Relative Importance Contribution (%)', fontsize=12)
plt.ylabel('Feature', fontsize=12)
for i, v in enumerate(importance_series.values):
    plt.text(v + 0.5, i, f"{v:.1f}%\修, va='center', fontsize=11, fontweight='semibold')
plt.xlim(0, max(importance_series.values) + 5)
plt.tight_layout()
plt.show()

print("Top Pricing Factors:")
for rank, (feat, pct) in enumerate(importance_series.items(), 1):
    print(f"  {rank}. {feat:<18}: {pct:.2f}%")

---
### Step 7: Inference Pipeline: Price Prediction for Sample Cars

Build a clean, reusable inference function that receives raw vehicle attributes, applies data cleaning, imputes missing mechanical specs, scales features, and predicts fair asking prices with confidence intervals.

In [ ]:
# Precompute brand-model mechanical medians for automatic imputation during inference
grouped_specs = df_clean.groupby(['brand', 'model'])[['mileage', 'engine', 'max_power', 'seats']].median()
brand_model_specs = {}
for (b, m), r in grouped_specs.iterrows():
    b_str, m_str = str(b), str(m)
    if b_str not in brand_model_specs:
        brand_model_specs[b_str] = {}
    brand_model_specs[b_str][m_str] = {
        'mileage': float(r['mileage']),
        'engine': float(r['engine']),
        'max_power': float(r['max_power']),
        'seats': int(r['seats'])
    }

global_defaults = {
    'mileage': float(df_clean['mileage'].median()),
    'engine': float(df_clean['engine'].median()),
    'max_power': float(df_clean['max_power'].median()),
    'seats': int(df_clean['seats'].median())
}

def predict_car_price(
    brand: str,
    model: str,
    vehicle_age: int,
    km_driven: int,
    seller_type: str = "Individual",
    fuel_type: str = "Petrol",
    transmission_type: str = "Manual",
    mileage: float = None,
    engine: float = None,
    max_power: float = None,
    seats: int = 5
) -> dict:
    """
    Reusable CarX Inference Pipeline:
    Accepts raw vehicle attributes, applies data cleaning & imputation,
    and outputs estimated selling price in INR with confidence bounds.
    """
    b_norm = brand.strip().title()
    if 'Maruti' in b_norm:
        b_norm = 'Maruti'
    m_norm = model.strip()
    
    # Fallback to model defaults if mechanical attributes are not supplied
    specs = brand_model_specs.get(b_norm, {}).get(m_norm, global_defaults)
    eff_mileage = float(mileage if mileage is not None else specs.get('mileage', 19.5))
    eff_engine = float(engine if engine is not None else specs.get('engine', 1200.0))
    eff_power = float(max_power if max_power is not None else specs.get('max_power', 82.0))
    eff_seats = int(seats if seats is not None else specs.get('seats', 5))
    
    # Prepare feature dataframe matching training pipeline
    input_df = pd.DataFrame([{
        'vehicle_age': int(vehicle_age),
        'km_driven': int(km_driven),
        'mileage': eff_mileage,
        'engine': eff_engine,
        'max_power': eff_power,
        'seats': eff_seats,
        'brand': b_norm,
        'model': m_norm,
        'seller_type': seller_type,
        'fuel_type': fuel_type,
        'transmission_type': transmission_type
    }])
    
    best_model_pipe = trained_pipelines[best_algo]
    pred_log = best_model_pipe.predict(input_df)[0]
    est_price = round(float(np.expm1(pred_log)), -2)
    
    # Confidence bounds (±8% based on validation RMSE)
    price_low = round(est_price * 0.92, -2)
    price_high = round(est_price * 1.08, -2)
    
    return {
        'car': f"{b_norm} {m_norm}",
        'vehicle_age_years': vehicle_age,
        'odometer_km': km_driven,
        'fuel_transmission': f"{fuel_type} ({transmission_type})",
        'estimated_fair_price_inr': est_price,
        'price_range_low_inr': price_low,
        'price_range_high_inr': price_high,
        'formatted_price': f"₹{est_price:,.0f} (₹{est_price/100000:.2f} Lakhs)",
        'model_used': best_algo
    }

print("predict_car_price() inference pipeline function compiled successfully!")

#### 7.2 Run Predictions on Sample Cars Across Market Segments

In [ ]:
sample_cars = [
    {
        'brand': 'Maruti',
        'model': 'Swift',
        'vehicle_age': 4,
        'km_driven': 38000,
        'seller_type': 'Individual',
        'fuel_type': 'Petrol',
        'transmission_type': 'Manual'
    },
    {
        'brand': 'Hyundai',
        'model': 'Creta',
        'vehicle_age': 3,
        'km_driven': 32000,
        'seller_type': 'Dealer',
        'fuel_type': 'Diesel',
        'transmission_type': 'Automatic',
        'engine': 1493,
        'max_power': 113.4,
        'mileage': 18.5
    },
    {
        'brand': 'Honda',
        'model': 'City',
        'vehicle_age': 6,
        'km_driven': 55000,
        'seller_type': 'Individual',
        'fuel_type': 'Petrol',
        'transmission_type': 'Manual',
        'engine': 1498,
        'max_power': 117.3,
        'mileage': 17.8
    },
    {
        'brand': 'Mercedes-Benz',
        'model': 'C-Class',
        'vehicle_age': 5,
        'km_driven': 42000,
        'seller_type': 'Dealer',
        'fuel_type': 'Diesel',
        'transmission_type': 'Automatic',
        'engine': 1950,
        'max_power': 192.0,
        'mileage': 12.6
    }
]

sample_results = []
for car in sample_cars:
    res = predict_car_price(**car)
    sample_results.append(res)

df_sample_output = pd.DataFrame(sample_results)
display(df_sample_output[['car', 'vehicle_age_years', 'odometer_km', 'fuel_transmission', 'formatted_price', 'model_used']])

---
### Step 8: Model Artifacts Export
Persist all model artifacts for production serving:
1. `valuation_model.joblib`: Serialized scikit-learn Pipeline
2. `valuation_model_light.json`: Lightweight pure-Python decision tree structure for zero-dependency Vercel Serverless execution
3. `baseline_lookup.json`: Baseline hierarchical price lookup
4. `valuation_metadata.json`: Model version, training metrics, and feature configurations

In [ ]:
models_dir = Path('models')
models_dir.mkdir(parents=True, exist_ok=True)

# 1. Save Joblib Pipeline
best_pipe = trained_pipelines[best_algo]
joblib.dump(best_pipe, models_dir / 'valuation_model.joblib')
print(f"✅ Saved Joblib Pipeline to {models_dir / 'valuation_model.joblib'}")

# 2. Save Baseline Lookup
with open(models_dir / 'baseline_lookup.json', 'w', encoding='utf-8') as f:
    json.dump(baseline_table, f, indent=2)
print(f"✅ Saved Baseline Lookup to {models_dir / 'baseline_lookup.json'}")

# 3. Export Lightweight Pure-Python Model JSON (for Vercel Serverless)
if best_algo == 'XGBoost':
    booster = best_pipe.named_steps['regressor'].get_booster()
    raw_trees = [json.loads(t) for t in booster.get_dump(dump_format='json')]
    scaler = best_pipe.named_steps['preprocessor'].named_transformers_['num']
    ohe = best_pipe.named_steps['preprocessor'].named_transformers_['cat']
    
    light_model_data = {
        'model_type': 'xgboost_pure_python',
        'base_score': 0.5,
        'trees': raw_trees,
        'num_features': NUMERICAL_FEATURES,
        'cat_features': CATEGORICAL_FEATURES,
        'scaler_mean': scaler.mean_.tolist(),
        'scaler_scale': scaler.scale_.tolist(),
        'categories': [c.tolist() for c in ohe.categories_],
        'brand_model_specs': brand_model_specs,
        'global_specs': global_defaults
    }
    with open(models_dir / 'valuation_model_light.json', 'w', encoding='utf-8') as f:
        json.dump(light_model_data, f)
    print(f"✅ Saved Lightweight Model to {models_dir / 'valuation_model_light.json'} ({len(raw_trees)} trees)")

# 4. Save Metadata
metadata = {
    'model_version': f"v1.0-{best_algo.lower()}",
    'selected_algorithm': best_algo,
    'training_timestamp': datetime.now().isoformat(),
    'total_records': len(df_clean),
    'features': {'numerical': NUMERICAL_FEATURES, 'categorical': CATEGORICAL_FEATURES},
    'feature_importances': importance_series.to_dict(),
    'metrics': benchmark_results[best_algo]
}
with open(models_dir / 'valuation_metadata.json', 'w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=2)
print(f"✅ Saved Metadata to {models_dir / 'valuation_metadata.json'}")